In [3]:
import pandas as pd
import useful_functions as uf

df = pd.DataFrame({'a': [1, 2], 'b': [3, 4]})
uf.show_df_info(df)  # This should print DataFrame info


First few rows of the DataFrame:
   a  b
0  1  3
1  2  4

DataFrame columns:
['a', 'b']

DataFrame shape: (2, 2)

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   a       2 non-null      int64
 1   b       2 non-null      int64
dtypes: int64(2)
memory usage: 164.0 bytes
None


In [34]:
#benchmark_df = pd.read_csv('../data/raw/synthetic_logs_10M.csv')
benchmark_df = pd.read_parquet('../data/raw/synthetic_logs_10M.parquet')
#uf.show_df_info(benchmark_df)

In [30]:
# Display unique values in columns that might be used as categorical
benchmark_df[['source_ip', 'destination_ip', 'port', 'protocol', 'event_type', 'severity','user', 'status_code', 'country', 'device_type' ]].nunique()

source_ip          9
destination_ip     7
port               9
protocol           3
event_type         8
severity           4
user               9
status_code        6
country           10
device_type        8
dtype: int64

In [31]:
# Check what type bytes should use
rt_bytes = uf.get_safe_int_type(benchmark_df['bytes'])
rt_ms = uf.get_safe_int_type(benchmark_df['response_time_ms'])
rt_session = uf.get_safe_int_type(benchmark_df['session_id'])

print(f"Recommended type for session_id: {rt_session}")
print(f"Recommended type for response_time_ms: {rt_ms}")
print(f"Recommended type for bytes: {rt_bytes}")

Recommended type for session_id: uint32
Recommended type for response_time_ms: uint16
Recommended type for bytes: uint32


In [ ]:
from useful_functions import optimize_df_types
def optimize_benchmark_df(bdf):
    customer_types = {
        'datetime64[ns]': ['timestamp'],
        'category': ['source_ip', 'destination_ip', 'port', 'protocol', 'event_type', 'severity','user', 'status_code', 'country', 'device_type'],
        'uint32': ['bytes', 'session_id'],
        'uint16': ['response_time_ms'],
        'float32': ['risk_score'] 
    }
    return optimize_df_types(bdf, customer_types)

In [ ]:
#uf.print_optimization_report(benchmark_df, new_benchmark_df, new_customer_types)

DataFrame Memory Usage Optimization Report

---- Per-column memory usage and dtype ----
                  Before (bytes)  After (bytes)  Delta (bytes) Before dtype  \
Index                        132            132              0          NaN   
Total                 5786322699      320006138     5466316561                
bytes                   80000000       40000000       40000000        int64   
country                510000000       10000810      499999190       object   
destination_ip         602862767       10000722      592862045       object   
device_type            594890849       10000764      584890085       object   
event_type             618892727       10000800      608891927       object   
port                    80000000       10000372       69999628        int64   
protocol               520499601       10000265      510499336       object   
response_time_ms        80000000       20000000       60000000        int64   
risk_score              80000000       4000

In [35]:
# Before optimization
print("Before optimization:")
old_mem_usage = benchmark_df.memory_usage(deep=True).sum() / 1024
benchmark_df.info(memory_usage='deep')

# Apply optimization
new_customer_types = {
    'datetime64[ns]': ['timestamp'],
    'category': ['source_ip', 'destination_ip', 'port', 'protocol', 'event_type', 'severity','user', 'status_code', 'country', 'device_type'],
    'uint32': ['bytes', 'session_id'],
    'uint16': ['response_time_ms'],
    'float32': ['risk_score'] # Acceptable loss for fraud detection. It might need to be float64 for other applications
}
# Replacing benchmark_df with the optimized version to save memory
benchmark_df = uf.optimize_df_types(benchmark_df, new_customer_types)

# After optimization  

new_mem_usage = benchmark_df.memory_usage(deep=True).sum() / 1024
print("\nAfter optimization:")
benchmark_df.info(memory_usage='deep')

Before optimization:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000000 entries, 0 to 9999999
Data columns (total 15 columns):
 #   Column            Dtype  
---  ------            -----  
 0   timestamp         object 
 1   source_ip         object 
 2   destination_ip    object 
 3   port              int64  
 4   protocol          object 
 5   event_type        object 
 6   severity          object 
 7   user              object 
 8   status_code       int64  
 9   bytes             int64  
 10  response_time_ms  int64  
 11  country           object 
 12  device_type       object 
 13  session_id        int64  
 14  risk_score        float64
dtypes: float64(1), int64(5), object(9)
memory usage: 5.4 GB

After optimization:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000000 entries, 0 to 9999999
Data columns (total 15 columns):
 #   Column            Dtype         
---  ------            -----         
 0   timestamp         datetime64[ns]
 1   source_ip         categ